<a href="https://colab.research.google.com/github/whateveri/models/blob/goScript_TinyVGG/general_scriptMode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Get Data

In [1]:
import os
import zipfile
from pathlib import Path
import requests

data_path=Path("data/")
image_path=data_path / "pizza_steak_sushi"

URL_="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip"

if image_path.is_dir():
    print(f'directory {image_path} already exists')

else:
    print(f'directory {image_path} not exist,creating one....')
    image_path.mkdir(parents=True,exist_ok=True)

    with open(data_path / "pizza_steak_sushi.zip",mode='wb') as f:
        print("downloading food data...")
        request=requests.get(url=URL_)
        f.write(request.content)

    with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip",mode='r') as zip:
        print('Extracting zip file....')
        zip.extractall(path=image_path)

    os.remove(path=data_path / "pizza_steak_sushi.zip")


directory data/pizza_steak_sushi not exist,creating one....
downloading food data...
Extracting zip file....


In [2]:
!mkdir generalModel

# Create dataset and dataloader

In [3]:
%%writefile generalModel/data_setup.py

import os
from torchvision import transforms,datasets
from torch.utils.data import DataLoader

NUM_WORKERS=os.cpu_count()

def create_dataloaders(train_dir:str,
                       test_dir:str,
                       train_transform:transforms.Compose,
                       test_transform:transforms.Compose,
                       batch_size:int,
                       num_workers=NUM_WORKERS):
    train_data=datasets.ImageFolder(root=train_dir,
                                    transform=train_transform)
    test_data=datasets.ImageFolder(root=test_dir,
                                    transform=test_transform)

    class_names=train_data.classes

    train_dataloader=DataLoader(dataset=train_data,
                                batch_size=batch_size,
                                shuffle=True,
                                num_workers=NUM_WORKERS,
                                pin_memory=True)
    test_dataloader=DataLoader(dataset=test_data,
                               batch_size=batch_size,
                               shuffle=False,
                               num_workers=NUM_WORKERS,
                               pin_memory=True)
    return train_dataloader,test_dataloader,class_names

Writing generalModel/data_setup.py


# Model Building(TinyVGG)

In [4]:
%%writefile generalModel/model_builder.py

import torch
from torch import nn

class TinyVGG(nn.Module):
    def __init__(self,input_features:int,
                 hidden_features:int,
                 output_features:int
                ) -> None:
        super().__init__()

        self.block1=nn.Sequential(
            nn.Conv2d(in_channels=input_features,
                      out_channels=hidden_features,
                      kernel_size=3,
                      stride=1,
                      padding=0),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_features,
                      out_channels=hidden_features,
                      kernel_size=3,
                      stride=1,
                      padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )

        self.block2=nn.Sequential(
            nn.Conv2d(in_channels=hidden_features,
                      out_channels=hidden_features,
                      kernel_size=3,
                      stride=1,
                      padding=0),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_features,
                      out_channels=hidden_features,
                      kernel_size=3,
                      stride=1,
                      padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2)

        )

        self.classifier=nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=13*13*hidden_features,
                      out_features=output_features)
        )

    def forward(self,x:torch.Tensor):
        x=self.block1(x)
        x=self.block2(x)
        x=self.classifier(x)

        return x


Writing generalModel/model_builder.py


# Train and Evaluate

In [5]:
%%writefile generalModel/eigine.py
import torch
from tqdm.auto import tqdm
from typing import List,Dict,Tuple


def train_step(model:nn.Module,
               dataloader:torch.utils.data.DataLoader,
               loss_fn:nn.Module,
               optimizer:torch.optim.Optimizer,
               device:torch.device)-> Tuple[float,float]:
    model.train()

    train_loss, train_acc=0.0,0.0

    for batch,(X,y) in enumerate(dataloader):
        X,y=X.to(device),y.to(device)

        y_pred_logit=model(X)

        loss=loss_fn(y_pred_logit,y)

        train_loss+=loss.item()

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        y_pred_class=torch.argmax(torch.softmax(y_pred_logit,dim=1),dim=1)

        train_acc+=((y_pred_class==y).sum().item() / len(y_pred_logit))

    train_loss=train_loss/len(dataloader)
    train_acc=train_acc/len(dataloader)

    return train_loss,train_acc


def test_step(model:nn.Module,
              dataloader:torch.utils.data.DataLoader,
              loss_fn:nn.Module,
              optimizer:torch.optim.Optimizer,
              device:torch.device)->Tuple[float,float]:
    model.eval()
    test_loss,test_acc=0.0,0.0

    with torch.inference_mode():
        for batch,(X,y) in enumerate(dataloader):
            X,y=X.to(device),y.to(device)

            y_pred_logit=model(X)

            loss=loss_fn(y_pred_logit,y)

            test_loss+=loss.item()

            y_pred_label=torch.argmax(y_pred_logit,dim=1)

            test_acc+=((y_pred_label==y).sum().item() / len(y_pred_logit))

    test_loss=test_loss / len(dataloader)
    test_acc=test_acc / len(dataloader)

    return test_loss,test_acc


def train(model:nn.Module,
          train_loader:torch.utils.data.DataLoader,
          test_loader:torch.utils.data.DataLoader,
          loss_fn:nn.Module,
          optimizer:torch.optim.Optimizer,
          epochs:int,
          device:torch.device
          )-> Dict[str,int]:

    results={
        "train_losses":[],
        "train_acces":[],
        "test_losses":[],
        "test_acces":[]
    }

    for epoch in tqdm(range(epochs)):

        train_loss,train_acc=train_step(model=model,
                                            dataloader=train_loader,
                                            loss_fn=loss_fn,
                                            optimizer=optimizer,
                                            device=device)

        test_loss,test_acc=test_step(model=model,
                                         dataloader=test_loader,
                                         loss_fn=loss_fn,
                                         optimizer=optimizer,
                                         device=device)
        print(f'Epoch {epoch+1} |'
              f'Train loss: {train_loss}'
              f'Train acc: {train_acc}'
              f'Test loss: {test_loss}'
              f'Test acc: {test_acc}')
        results["train_losses"].append(train_loss)
        results["train_acces"].append(train_acc)
        results["train_losses"].append(test_loss)
        results["train_losses"].append(test_acc)

    return results














Writing generalModel/eigine.py


# Save model

In [6]:
%%writefile generalModel/utils.py

import torch
from pathlib import Path




def save_model(model:torch.nn.Module,
               path:str,
               model_name:str):
    model_dir=Path(path)
    model_dir.mkdir(parents=True,
                    exist_ok=True)

    assert model_name.endswith("pth") or model_name.endswith("pt"), "model name should not ends with 'pt' or 'pth'"
    model_save_path=model_dir / model_name

    print(f'Model already saved in {model_save_path}')

    torch.save(obj=model.state_dict(),f=model_save_path)

Writing generalModel/utils.py


In [8]:
%%writefile generalModel/train.py

import os
import torch
import data_setup,eigine,utils,model_builder

from torchvision import transforms

NUM_EPOCHS=5
BATCH_SIZE=32
HIDDEN_UNITS=10
lr=0.001

train_dir="data/pizza_steak_sushi/train"
test_dir="data/pizza_steak_sushi/test"

device="cuda" if torch.cuda.is_available() else "cpu"

data_trainsform=transforms.Compose([
    transforms.Resize(size=(64,64)),
    transforms.ToTensor()
])


train_dataloader,test_dataloader,class_names=data_setup.create_dataloaders(                      train_dir=train_dir,
                       test_dir=test_dir,
                       train_transform=data_trainsform,
                       test_transform=data_trainsform,
                       batch_size=BATCH_SIZE
                       )

model=model_builder.TinyVGG(input_features=3,
                 hidden_features=HIDDEN_UNITS,
                 output_features=len(class_names)).to(device)

loss_fn=torch.nn.CrossEntropyLoss()

optimizer=torch.optim.Adam(params=model.parameters(),
                           lr=lr)

eigine.train(model=model,
             train_loader=train_dataloader,
             test_loader=test_dataloader,
             loss_fn=loss_fn,
             optimizer=optimizer,
             epochs=NUM_EPOCHS,
             device=device)

utils.save_model(model=model,
               path="models",
               model_name="go_script_TinyVGG")





Writing generalModel/train.py


In [9]:
from google.colab import files
import shutil

# 将整个当前工作目录打包为 zip 文件（不包括 /content/sample_data）
shutil.make_archive("colab_folder", 'zip', ".")

# 下载打包好的 zip 文件
files.download("colab_folder.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>